In [ ]:
#  Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, classification_report,
                             RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve


In [ ]:
data = pd.read_csv("train (1).csv")
print(data.shape)
data.head()


In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
data.describe()

📊 Exploratory Data Analysis (EDA)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
g_labels = ['Male', 'Female']
c_labels = ['No', 'Yes']
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]])
fig.add_trace(go.Pie(labels=g_labels, values=data['gender'].value_counts(), name="Gender"),
              1, 1)
fig.add_trace(go.Pie(labels=c_labels, values=data['Class/ASD'].value_counts(), name="Class/ASD"),
              1, 2)


fig.update_traces(hole=.4, hoverinfo="label+percent+name", textfont_size=16)

fig.update_layout(
    title_text="Gender and Class/ASD Distributions",

    annotations=[dict(text='Gender', x=0.19, y=0.5, font_size=20, showarrow=False),
                 dict(text='Class/ASD', x=0.82, y=0.5, font_size=18, showarrow=False)])
fig.show()


#The distribution between males and females is similar, but the response rate with ASD varies by gender.

#The ASD category is unevenly distributed → There is a possibility of an imbalance problem.

The distribution between males and females is similar, but the response rate with ASD varies by gender.

The ASD category is unevenly distributed → There is a possibility of an imbalance problem.

In [ ]:

import plotly.express as px

# Age distribution
plt.figure(figsize=(8,5))
sns.histplot(data['age'], kde=True, bins=20, color="skyblue")
plt.title("Age Distribution")
plt.show()


Most cases occur in the young age group (10–20 years).

Age may still be an important prognostic feature.

In [ ]:

# Gender distribution
fig = px.histogram(data, x="gender", color="Class/ASD", barmode="group",
                   title="Gender Distribution by ASD Class")
fig.show()


Insight : The dataset is slightly imbalanced in gender distribution, with more males than females

In [ ]:


# Ethnicity distribution
ethnicity_count = data['ethnicity'].value_counts().head(10)
plt.figure(figsize=(8,5))
sns.barplot(x=ethnicity_count.values, y=ethnicity_count.index, palette="viridis")
plt.title("Top 10 Ethnicities")
plt.show()

Insight : The majority of samples belong to one or two dominant ethnic groups, which may bias the model

In [ ]:

# Country of residence
fig = px.choropleth(data, locations="contry_of_res", locationmode="country names",
                    color="Class/ASD", title="ASD Cases by Country")
fig.show()


In [ ]:

# Jaundice
sns.countplot(x="jaundice", hue="Class/ASD", data=data, palette="Set2")
plt.title("Jaundice History vs ASD")
plt.show()

Insight : Only a small portion of individuals report jaundice, showing class imbalance in this feature


In [ ]:

# Autism in family
sns.countplot(x="austim", hue="Class/ASD", data=data, palette="coolwarm")
plt.title("Family Autism History vs ASD")
plt.show()

Insight : The majority of participants report no family history of autism, suggesting this is a less common trait in the dataset


In [ ]:


# Used app before
fig = px.pie(data, names="used_app_before", title="Used App Before Distribution")
fig.show()


In [ ]:

# Relation
plt.figure(figsize=(10,5))
sns.countplot(y="relation", data=data, order=data['relation'].value_counts().index, palette="pastel")
plt.title("Relation to Test Taker")
plt.show()

In [ ]:


score_cols = [f"A{i}_Score" for i in range(1,11)]

# Heatmap
plt.figure(figsize=(10,6))
sns.heatmap(data[score_cols + ['result']].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap of Screening Scores")
plt.show()


Insight Screening scores (A1–A10) are strongly correlated, confirming they capture similar behavioral patterns


In [ ]:

# Radar chart (avg per class)
import plotly.graph_objects as go
avg_scores = data.groupby("Class/ASD")[score_cols].mean().T
fig = go.Figure()
for label in avg_scores.columns:
    fig.add_trace(go.Scatterpolar(r=avg_scores[label].values,
                                  theta=avg_scores.index,
                                  fill="toself", name=label))
fig.update_layout(title="Average Screening Scores by ASD Class",
                  polar=dict(radialaxis=dict(visible=True)))
fig.show()


In [ ]:





# Class distribution
sns.countplot(x="Class/ASD", data=data, palette="muted")
plt.title("ASD Class Distribution")
plt.show()

# Gender × Class
sns.countplot(x="gender", hue="Class/ASD", data=data, palette="Set1")
plt.title("Gender vs ASD")
plt.show()

# Age × Class
plt.figure(figsize=(8,5))
sns.violinplot(x="Class/ASD", y="age", data=data, palette="Set2")
plt.title("Age vs ASD Class")
plt.show()

# Sunburst
fig = px.sunburst(data, path=["gender","ethnicity","Class/ASD"],
                  title="ASD Distribution by Gender & Ethnicity")
fig.show()


Data PreProcessing

In [ ]:
data=data.drop(["ID"] , axis=1)

In [ ]:
data["gender"] = data["gender"].replace({"f": 0, "m": 1})

In [ ]:
data = pd.get_dummies(data, columns=['ethnicity','jaundice',
       'austim', 'contry_of_res', 'used_app_before', 'age_desc',
       'relation'] , dtype=int)

In [ ]:
data.dtypes

In [ ]:
y=data["Class/ASD"]
x=data.drop(["Class/ASD"] , axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler=StandardScaler()
x=scaler.fit_transform(x)


In [ ]:
x_train , x_test , y_train , y_test =train_test_split(x ,y , test_size=0.3,random_state=42 , stratify=y)
print(x_train.shape , y_train.shape) , x_test , y_test

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


modeles = {
    "RandomForestClassifier": RandomForestClassifier(random_state=42),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=500),
    "XGBClassifier": XGBClassifier(use_label_encoder=False, eval_metric="logloss"),
    "SVC": SVC(probability=True) 
}


parameters = {
    "RandomForestClassifier": {
        "n_estimators": [100, 200],
        "max_depth": [None, 5, 10],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    },
    "GradientBoostingClassifier": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0]
    },
    "LogisticRegression": {
        "penalty": ["l1", "l2"],
        "C": [0.1, 1, 10],
        "solver": ["liblinear", "saga"],
        "max_iter": [200, 500]
    },
    "XGBClassifier": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    "SVC": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale", "auto"]
    }
}

# Training + Evaluation
for name, model in modeles.items():
    print(f"\n{name}")

   
    grid = GridSearchCV(model, parameters[name], cv=3, scoring="accuracy", n_jobs=-1)
    grid.fit(x_train, y_train)

    best_model = grid.best_estimator_
    y_pre = best_model.predict(x_test)

    accuracy = accuracy_score(y_test, y_pre)
    print(f"{name} - Best Params: {grid.best_params_}")
    print(f"{name} - Accuracy Score: {accuracy:.4f}")

    # Visualizations
    RocCurveDisplay.from_estimator(best_model, x_test, y_test)
    plt.title(f"ROC Curve: {name}")
    plt.show()

    PrecisionRecallDisplay.from_estimator(best_model, x_test, y_test)
    plt.title(f"Precision-Recall Curve: {name}")
    plt.show()

    ConfusionMatrixDisplay.from_estimator(best_model, x_test, y_test)
    plt.title(f"Confusion Matrix: {name}")
    plt.show()


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
importances_dict = {}
best_model = None
best_score = 0

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in modeles.items():
    # Cross-validation (F1 macro)
    cv_scores = cross_val_score(model, x, y, cv=cv, scoring="f1_macro")
    print(f"{name} CV F1 mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    # Train on full train data
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Test Accuracy: {acc:.3f}")

    # Track best model
    if acc > best_score:
        best_score = acc
        best_model = model

    # Feature importance (if available)
    if hasattr(model, "feature_importances_"):
        importances_dict[name] = model.feature_importances_
    elif hasattr(model, "coef_"):
        importances_dict[name] = np.abs(model.coef_[0])


#  Feature Importance Plots

feature_names = data.drop("Class/ASD", axis=1).columns

for name, imp in importances_dict.items():
    sorted_idx = np.argsort(imp)[-15:]  
    plt.figure(figsize=(8,5))
    sns.barplot(x=imp[sorted_idx], y=feature_names[sorted_idx], palette="viridis")
    plt.title(f"Top Features - {name}")
    plt.show()


#  Save Best Model

import pickle
with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print(f"Best model saved: {best_model}")


In [ ]:
for name,a in modeles.items():
    train_sizes, train_scores, val_scores = learning_curve(
        a, x, y, cv=5, scoring='f1', train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1)
    plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train')
    plt.plot(train_sizes, val_scores.mean(axis=1), marker='o', label='Validation')
    plt.xlabel('Training examples'); plt.ylabel('F1'); plt.title(f'Learning Curve:{name}'); plt.legend(); plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
# Calibration Curve
for name,a in modeles.items():
    prob_pos = a.predict_proba(x_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, prob_pos, n_bins=10, strategy='uniform')
    plt.plot(mean_pred, frac_pos, marker='o'); plt.plot([0,1],[0,1],'--')
    plt.xlabel('Predicted probability'); plt.ylabel('True fraction of positives'); plt.title(f'Calibration{name}'); plt.show()

DeepLearning

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn import metrics
model1 = tf.keras.Sequential([
        tf.keras.layers.Input(shape=[94])    ,
        
         tf.keras.layers.Dense(units=64 , activation=tf.nn.leaky_relu),
         
         tf.keras.layers.Dense(units=128 , activation=tf.nn.leaky_relu,),
         
         tf.keras.layers.Dense(units=1 , activation=tf.nn.sigmoid)
])
model1.summary()

In [ ]:
callback = keras.callbacks.EarlyStopping(monitor='loss',
patience=3)
model1.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])

hist=model1.fit(x=x_train , y=y_train, epochs=50 , callbacks=[callback])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

# Loss Curve
plt.subplot(1,2,1)
plt.plot(hist.history['loss'], label='Train Loss')
if 'val_loss' in hist.history:
    plt.plot(hist.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

# Accuracy Curve
plt.subplot(1,2,2)
plt.plot(hist.history['accuracy'], label='Train Accuracy')
if 'val_accuracy' in hist.history:
    plt.plot(hist.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.legend()

plt.show()


In [ ]:
test_loss, test_acc = model1.evaluate(x_test, y_test, verbose=0)
print(f"📊 Test Accuracy: {test_acc:.4f}")
print(f"📉 Test Loss: {test_loss:.4f}")
